# Balance Scenario: Original vs Contact-Reward — MAPPO Comparison

This notebook trains MAPPO on two variants of the VMAS Balance scenario and compares results.

**Runtime**: Select `Runtime → Change runtime type → T4 GPU` for best performance.

**Prerequisites**: Upload the `Code/` folder to `/content/Code/` (or to Google Drive and mount it).

## 1. Install Dependencies

In [ ]:
!pip install -q vmas torchrl tensordict benchmarl matplotlib pandas

## 2. Check GPU & Verify Imports

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    DEVICE = "cuda"
else:
    print("No GPU detected — training will run on CPU (slower).")
    DEVICE = "cpu"

import vmas
import torchrl
print(f"VMAS: {vmas.__version__}")
print(f"TorchRL: {torchrl.__version__}")
print(f"\nUsing device: {DEVICE}")

## 3. Verify Code Folder

In [ ]:
import os

# Adjust this path if you uploaded elsewhere or mounted Google Drive
CODE_DIR = "/content/Code"

# Check if the Code folder exists; if mounted via Drive, adjust:
# CODE_DIR = "/content/drive/MyDrive/Code"

assert os.path.isdir(CODE_DIR), f"Code folder not found at {CODE_DIR}. Upload it or update the path."
print("Contents of Code/:")
for item in sorted(os.listdir(CODE_DIR)):
    print(f"  {item}")

print("\nScenarios:")
for item in sorted(os.listdir(os.path.join(CODE_DIR, "scenarios"))):
    print(f"  {item}")

print("\n✓ Code folder found and verified.")

## 4. Training Configuration

Edit these values to adjust training. The defaults run in ~20-30 min total on a T4 GPU.

In [ ]:
# ── Training hyperparameters ──
N_AGENTS = 3
MAX_STEPS = 200           # Steps per episode
NUM_ENVS = 128            # Parallel environments (increase on GPU)
TOTAL_FRAMES = 300_000    # Total training frames per run
FRAMES_PER_BATCH = 6_000  # Frames collected before each PPO update
MINIBATCH_SIZE = 400
N_MINIBATCH_ITERS = 4
LR = 5e-4
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_EPSILON = 0.2
ENTROPY_COEF = 0.01
CRITIC_COEF = 1.0
MAX_GRAD_NORM = 5.0
HIDDEN_DIM = 256
SHARE_PARAMS = True
CENTRALISED_CRITIC = True

# Contact-reward coefficient (for the 'active' scenario)
CONTACT_REWARD_COEFF = 0.5

# Seeds for comparison
SEEDS = [0, 1, 2]

print(f"Config: {N_AGENTS} agents, {NUM_ENVS} envs, {TOTAL_FRAMES:,} frames, device={DEVICE}")
print(f"Estimated time per run: ~3-5 min (T4 GPU) / ~15-30 min (CPU)")
print(f"Total: {len(SEEDS)} seeds × 2 scenarios = {len(SEEDS)*2} runs")

## 5. Training Engine

This cell defines the full MAPPO training loop. Run it to define the functions; actual training is triggered in the next cell.

In [ ]:
import csv
import importlib.util
import time
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
from torch import nn

from torchrl.collectors import SyncDataCollector
from torchrl.data import TensorDictReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.envs import RewardSum, TransformedEnv
from torchrl.envs.libs.vmas import VmasEnv
from torchrl.envs.utils import ExplorationType, set_exploration_type
from torchrl.modules import MultiAgentMLP, ProbabilisticActor, TanhNormal
from torchrl.objectives import ClipPPOLoss, ValueEstimators
from tensordict.nn import TensorDictModule, TensorDictSequential

SCENARIOS_DIR = Path(CODE_DIR) / "scenarios"
RESULTS_DIR = Path(CODE_DIR) / "results"


def _load_scenario_class(scenario_name):
    fmap = {
        "original": SCENARIOS_DIR / "balance_original.py",
        "active":   SCENARIOS_DIR / "balance_active.py",
    }
    path = fmap[scenario_name]
    spec = importlib.util.spec_from_file_location(f"balance_{scenario_name}", path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.Scenario


def make_env(scenario_name, num_envs, device, seed, max_steps, n_agents,
             continuous_actions=True, **scenario_kwargs):
    ScenarioCls = _load_scenario_class(scenario_name)
    env = VmasEnv(
        scenario=ScenarioCls(),
        num_envs=num_envs,
        device=device,
        seed=seed,
        continuous_actions=continuous_actions,
        max_steps=max_steps,
        n_agents=n_agents,
        categorical_actions=True,
        clamp_actions=True,
        **scenario_kwargs,
    )
    env = TransformedEnv(
        env,
        RewardSum(in_keys=[env.reward_key], out_keys=[("agents", "episode_reward")])
    )
    return env


def build_policy_and_critic(env, hidden_dim, share_params, centralised_critic):
    n_agents = len(env.agents)
    obs_spec = env.full_observation_spec["agents", "observation"]
    act_spec = env.full_action_spec["agents", "action"]
    obs_size = obs_spec.shape[-1]
    act_size = act_spec.shape[-1]

    policy_net = MultiAgentMLP(
        n_agent_inputs=obs_size,
        n_agent_outputs=2 * act_size,
        n_agents=n_agents,
        centralised=False,
        share_params=share_params,
        depth=2,
        num_cells=hidden_dim,
        activation_class=nn.Tanh,
    )
    policy_module = TensorDictModule(
        policy_net,
        in_keys=[("agents", "observation")],
        out_keys=[("agents", "loc_scale")],
    )
    # FIX: Use softplus + 1e-4 for scale to prevent negative/zero values
    unbind = TensorDictModule(
        lambda x: (
            x[..., :act_size],
            torch.nn.functional.softplus(x[..., act_size:]) + 1e-4,
        ),
        in_keys=[("agents", "loc_scale")],
        out_keys=[("agents", "loc"), ("agents", "scale")],
    )
    policy = ProbabilisticActor(
        module=TensorDictSequential(policy_module, unbind),
        in_keys=[("agents", "loc"), ("agents", "scale")],
        out_keys=[("agents", "action")],
        distribution_class=TanhNormal,
        distribution_kwargs={
            "low": act_spec.space.low,
            "high": act_spec.space.high,
        },
        return_log_prob=True,
        log_prob_key=("agents", "sample_log_prob"),
    )

    critic_net = MultiAgentMLP(
        n_agent_inputs=obs_size,
        n_agent_outputs=1,
        n_agents=n_agents,
        centralised=centralised_critic,
        share_params=share_params,
        depth=2,
        num_cells=hidden_dim,
        activation_class=nn.Tanh,
    )
    critic = TensorDictModule(
        critic_net,
        in_keys=[("agents", "observation")],
        out_keys=[("agents", "state_value")],
    )
    return policy, critic


class CSVLogger:
    def __init__(self, log_dir):
        os.makedirs(log_dir, exist_ok=True)
        self.path = os.path.join(log_dir, "progress.csv")
        self._file = None
        self._writer = None
        self._keys = None

    def log(self, data):
        if self._writer is None:
            self._keys = list(data.keys())
            self._file = open(self.path, "w", newline="")
            self._writer = csv.DictWriter(self._file, fieldnames=self._keys)
            self._writer.writeheader()
        self._writer.writerow({k: data.get(k, "") for k in self._keys})
        self._file.flush()

    def close(self):
        if self._file:
            self._file.close()


def train(scenario_name, seed, device=DEVICE):
    log_dir = str(RESULTS_DIR / scenario_name / f"seed_{seed}")
    os.makedirs(log_dir, exist_ok=True)
    logger = CSVLogger(log_dir)
    torch.manual_seed(seed)

    scenario_kwargs = {}
    if scenario_name == "active":
        scenario_kwargs["contact_reward_coeff"] = CONTACT_REWARD_COEFF

    env_fn = lambda: make_env(
        scenario_name=scenario_name, num_envs=NUM_ENVS, device=device,
        seed=seed, max_steps=MAX_STEPS, n_agents=N_AGENTS, **scenario_kwargs,
    )
    env = env_fn()
    n_agents_actual = len(env.agents)

    policy, critic = build_policy_and_critic(env, HIDDEN_DIM, SHARE_PARAMS, CENTRALISED_CRITIC)
    policy = policy.to(device)
    critic = critic.to(device)

    # FIX: Use double-f spelling + normalize_advantage_exclude_dims for multi-agent
    loss_module = ClipPPOLoss(
        actor_network=policy, critic_network=critic,
        clip_epsilon=CLIP_EPSILON, entropy_coeff=ENTROPY_COEF, critic_coeff=CRITIC_COEF,
        normalize_advantage=True,
        normalize_advantage_exclude_dims=(-2,),  # exclude agent dimension
    )
    loss_module.set_keys(
        reward=env.reward_key, action=("agents", "action"),
        done=("agents", "done"), terminated=("agents", "terminated"),
        sample_log_prob=("agents", "sample_log_prob"), value=("agents", "state_value"),
    )
    loss_module.make_value_estimator(ValueEstimators.GAE, gamma=GAMMA, lmbda=GAE_LAMBDA)
    optim = torch.optim.Adam(loss_module.parameters(), lr=LR)

    collector = SyncDataCollector(
        create_env_fn=env_fn, policy=policy,
        frames_per_batch=FRAMES_PER_BATCH, total_frames=TOTAL_FRAMES,
        device=device, storing_device=device,
    )
    replay_buffer = TensorDictReplayBuffer(
        storage=LazyTensorStorage(FRAMES_PER_BATCH, device=device),
        sampler=SamplerWithoutReplacement(), batch_size=MINIBATCH_SIZE,
    )

    total_frames_so_far = 0
    n_iters = 0
    t_start = time.time()
    best_reward = float("-inf")
    rewards_history = []
    total_batches = TOTAL_FRAMES // FRAMES_PER_BATCH

    print(f"\n{'='*60}")
    print(f"  MAPPO: {scenario_name} | seed={seed} | device={device}")
    print(f"{'='*60}")

    for batch in collector:
        n_iters += 1
        total_frames_so_far += batch.numel()

        # FIX: Expand root done/terminated to per-agent dimension
        # VMAS done is [batch, time, 1]; reward/value are [batch, time, n_agents, 1]
        for key in ["done", "terminated"]:
            root_val = batch.get(("next", key), None)
            if root_val is not None:
                expanded = root_val.unsqueeze(-2).expand(
                    *root_val.shape[:-1], n_agents_actual, 1
                )
                batch.set(("next", "agents", key), expanded)
            root_cur = batch.get(key, None)
            if root_cur is not None:
                expanded_cur = root_cur.unsqueeze(-2).expand(
                    *root_cur.shape[:-1], n_agents_actual, 1
                )
                batch.set(("agents", key), expanded_cur)

        # Compute GAE
        with torch.no_grad():
            loss_module.value_estimator(
                batch,
                params=loss_module.critic_network_params,
                target_params=loss_module.target_critic_network_params,
            )

        replay_buffer.extend(batch.reshape(-1))
        train_losses = []
        for _ in range(N_MINIBATCH_ITERS):
            for mb in replay_buffer:
                loss_vals = loss_module(mb)
                loss = loss_vals["loss_objective"] + loss_vals["loss_critic"] + loss_vals["loss_entropy"]
                # FIX: NaN guard — skip gradient update if loss is NaN/Inf
                if torch.isnan(loss) or torch.isinf(loss):
                    optim.zero_grad()
                    continue
                loss.backward()
                nn.utils.clip_grad_norm_(loss_module.parameters(), MAX_GRAD_NORM)
                optim.step()
                optim.zero_grad()
                train_losses.append(loss.item())

        mean_loss = np.mean(train_losses) if train_losses else float("nan")

        # FIX: Episode reward via batch["next"] (end-of-step done, not start-of-step)
        mean_reward = float("nan")
        try:
            next_td = batch.get("next")
            if next_td is not None:
                done_vals = next_td.get("done", None)
                ep_rew_vals = next_td.get(("agents", "episode_reward"), None)
                if done_vals is not None and ep_rew_vals is not None:
                    done_mask = done_vals.squeeze(-1)
                    if done_mask.any():
                        while ep_rew_vals.dim() > done_mask.dim() + 1:
                            ep_rew_vals = ep_rew_vals.squeeze(-1)
                        ep_rew_agent_mean = ep_rew_vals.mean(dim=-1)
                        mean_reward = ep_rew_agent_mean[done_mask].mean().item()
        except Exception:
            pass

        if not np.isnan(mean_reward) and mean_reward > best_reward:
            best_reward = mean_reward

        elapsed = time.time() - t_start
        fps = total_frames_so_far / max(elapsed, 1e-6)
        progress = total_frames_so_far / TOTAL_FRAMES * 100

        log_data = {
            "step": total_frames_so_far, "iteration": n_iters,
            "mean_reward": mean_reward, "best_reward": best_reward,
            "mean_loss": mean_loss,
            "fps": fps, "elapsed_s": elapsed,
        }
        logger.log(log_data)
        rewards_history.append((total_frames_so_far, mean_reward))

        if n_iters % 5 == 1 or total_frames_so_far >= TOTAL_FRAMES:
            eta_s = (TOTAL_FRAMES - total_frames_so_far) / max(fps, 1)
            print(f"  [{scenario_name:>8}] {progress:5.1f}%  iter={n_iters:4d}  "
                  f"frames={total_frames_so_far:>8,}  "
                  f"rew={mean_reward:>8.2f}  best={best_reward:>8.2f}  "
                  f"loss={mean_loss:.4f}  fps={fps:.0f}  "
                  f"ETA={eta_s/60:.1f}min")

    collector.shutdown()
    logger.close()
    env.close()

    print(f"  Done [{scenario_name}|seed={seed}] best={best_reward:.2f}  Logs → {log_dir}")
    return rewards_history

## 6. Run Training

This runs both scenarios across all seeds. Takes ~20-30 min on a T4 GPU.

In [ ]:
all_results = {}

for scenario_name in ["original", "active"]:
    all_results[scenario_name] = {}
    for seed in SEEDS:
        history = train(scenario_name, seed, device=DEVICE)
        all_results[scenario_name][seed] = history

print(f"\n{'='*60}")
print(f"  All training complete!")
print(f"{'='*60}")

## 7. Comparison Plot

Side-by-side reward curves averaged across seeds, with standard deviation shading.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def load_results_from_csv(scenario_name):
    """Load progress.csv files for all seeds of a scenario."""
    seed_data = {}
    for seed in SEEDS:
        csv_path = RESULTS_DIR / scenario_name / f"seed_{seed}" / "progress.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            seed_data[seed] = (df["step"].values, df["mean_reward"].values)
    return seed_data

def compute_mean_std(seed_data):
    if not seed_data:
        return None, None, None
    min_max_step = min(s[-1] for s, _ in seed_data.values())
    common_steps = np.linspace(0, min_max_step, 200)
    interped = [np.interp(common_steps, s, v) for s, v in seed_data.values()]
    interped = np.array(interped)
    return common_steps, np.nanmean(interped, axis=0), np.nanstd(interped, axis=0)

# ── Plot ──
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

configs = {
    "original": {"color": "#2196F3", "label": "Original Balance"},
    "active":   {"color": "#FF5722", "label": "Balance + Contact Reward"},
}

for name, meta in configs.items():
    seed_data = load_results_from_csv(name)
    if not seed_data:
        print(f"No data for {name}")
        continue
    steps, mean, std = compute_mean_std(seed_data)
    ax.plot(steps, mean, color=meta["color"], label=meta["label"], linewidth=2)
    ax.fill_between(steps, mean - std, mean + std, color=meta["color"], alpha=0.2)

ax.set_xlabel("Training Frames", fontsize=13)
ax.set_ylabel("Mean Episode Reward", fontsize=13)
ax.set_title("MAPPO on Balance: Original vs Contact-Reward\n"
             f"({len(SEEDS)} seeds, {N_AGENTS} agents, {NUM_ENVS} envs)",
             fontsize=15)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()

# Save
plots_dir = Path(CODE_DIR) / "plots"
plots_dir.mkdir(exist_ok=True)
fig.savefig(plots_dir / "comparison_reward.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nPlot saved to {plots_dir / 'comparison_reward.png'}")

## 8. Detailed Metrics Table

In [ ]:
print(f"{'='*65}")
print(f"  Final Performance (last 10% of training, mean ± std over seeds)")
print(f"{'='*65}")
print(f"  {'Scenario':<25} {'Mean Reward':<20} {'Seeds'}")
print(f"  {'-'*60}")

for name in ["original", "active"]:
    seed_data = load_results_from_csv(name)
    if not seed_data:
        print(f"  {name:<25} {'N/A':<20}")
        continue

    final_rewards = []
    for seed, (steps, rewards) in seed_data.items():
        # Average over last 10% of steps
        cutoff = int(0.9 * len(rewards))
        final_rewards.append(np.nanmean(rewards[cutoff:]))

    final_rewards = np.array(final_rewards)
    label = "Original" if name == "original" else "+ Contact Reward"
    print(f"  {label:<25} {np.mean(final_rewards):.2f} ± {np.std(final_rewards):.2f}       {list(seed_data.keys())}")

print()

## 9. Download Results

Run this cell to zip and download results + plots.

In [ ]:
import shutil

zip_path = shutil.make_archive("/content/balance_results", "zip", CODE_DIR, ".")
print(f"Zipped to: {zip_path}")

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not in Colab — find the zip at:", zip_path)